# Session 13.2: Bias-Variance, Cross-Validation & Regularisation

<table cellpadding="10" cellspacing="0" border="0" width="100%"><tr>
<td bgcolor="#ED1C24" width="4"></td>
<td bgcolor="#fce4ec">
<font color="#c62828"><b>Program:</b></font> Vishlesan i-Hub IIT Patna x Masai School -- AIM (AI & Machine Learning)<br>
<font color="#c62828"><b>Session ID:</b></font> 13.2 | <b>Week:</b> 13 | <b>Module:</b> Module 1 — Foundations of AI and Data Science<br>
<font color="#c62828"><b>Prerequisites:</b></font> Sessions 9.1, 9.2, 10.1, 11.x, 12.1, 12.2, 13.1<br>
<font color="#c62828"><b>Estimated completion time:</b></font> 90-120 minutes (self-paced)
</td>
</tr></table>

---

Vishlesan i-Hub IIT Patna × Masai School

## Learning Objectives

By the end of this notebook you will be able to:

1. Place a model on the bias-variance dartboard from its train/test error fingerprint.
2. Apply k-fold cross-validation via `cross_val_score` AND understand the manual loop underneath.
3. Recognise and fix the CV leakage trap by moving preprocessing inside a `Pipeline`.
4. Fit Ridge / Lasso / ElasticNet with sklearn and read coefficient-path plots.
5. Explain WHY L1 produces sparsity (the diamond corner argument).
6. Tune alpha via `RidgeCV`, `LassoCV`, and `GridSearchCV`.
7. Use Lasso for feature selection with `SelectFromModel`.


## Why this session matters — the Netflix Prize, 2009

In **October 2006** Netflix offered $1,000,000 to anyone who could improve their movie-recommendation algorithm by 10%. Three years later, in **September 2009**, a team called *BellKor's Pragmatic Chaos* won. Their winning entry was an ensemble of dozens of **regularised regression models**. The runner-up team, *The Ensemble*, technically tied them on the holdout set but lost on the timestamp tiebreaker — they submitted 20 minutes too late.

Both teams wrote in their post-mortems that *the single most important insight* was understanding that **better leaderboard score did not always mean better test score** — they had to design their cross-validation carefully to avoid overfitting the public leaderboard. The three concepts that decided that million dollars are exactly today's three concepts: **bias-variance, k-fold cross-validation, and regularised regression**.


## Setup & Imports

In [ ]:
# Pin minimums for reproducibility
!pip install -q -U "scikit-learn>=1.4" "plotly>=5.20" "joblib>=1.4"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 39.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

from sklearn.datasets import load_diabetes
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV,
)
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV,
)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import r2_score, mean_squared_error

RNG = 42
MASAI_RED = "#ED1C24"
BLUE = "#1565C0"
GREEN = "#2E7D32"
ORANGE = "#E65100"
PURPLE = "#7C4DFF"
import sklearn
print(f"Setup complete. sklearn {sklearn.__version__}")


Setup complete. sklearn 1.8.0


## The `box()` helper for styled callouts

In [ ]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

def box(kind, title, content):
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style=\"margin:12px 0; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;\">
    <strong style=\"color:{title_clr};\">{title}</strong><br>{content}
    </div>"""))


In [ ]:
box("industry", "$1M decided by these three concepts",
    "<b>Bias-variance.</b> BellKor's models were complex enough to capture real patterns but constrained to avoid memorising leaderboard noise.<br>"
    "<b>Cross-validation.</b> The 'leaderboard score' is a single test split. Teams that trusted it overfit to it. Teams that built honest internal CV beat them.<br>"
    "<b>Regularised regression.</b> Ridge in particular was the workhorse of the winning ensemble — it gave them coefficients that generalised, not coefficients that memorised.<br><br>"
    "Every concept in this notebook is one of the levers those teams pulled.")


## Part 0 — Load and explore the Diabetes dataset

10 features, 442 patients. Target = a quantitative measure of disease progression one year after baseline. All features are already mean-centered and scaled to unit variance — but we'll wrap StandardScaler in a Pipeline anyway to demonstrate the correct discipline (and to make our pipelines transferable to non-pre-scaled data).


In [ ]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target
feature_names = X.columns.tolist()
print(f"Shape: X = {X.shape}, y = {y.shape}")
print(f"Features: {feature_names}")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}], mean = {y.mean():.1f}")
X.head()


Shape: X = (442, 10), y = (442,)
Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target range: [25.0, 346.0], mean = 152.1


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [ ]:
box("definition", "Diabetes feature glossary",
    "age, sex, bmi, bp (blood pressure), s1-s6 (six blood serum measurements). "
    "All features have been mean-centered and scaled to unit variance by sklearn before distribution. "
    "The target is a 25-346 disease-progression score.")


In [ ]:
# Correlation heatmap of features
corr = X.corr()
fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=corr.round(2).values, texttemplate="%{text}",
    hovertemplate="%{x} vs %{y}: %{z:.2f}<extra></extra>",
))
fig.update_layout(
    title="Diabetes — feature correlations",
    paper_bgcolor="white", plot_bgcolor="white",
    width=650, height=550,
)
fig.show()


In [ ]:
box("output", "Note the s1-s6 correlations",
    "The six blood-serum measurements (s1 through s6) are partially correlated with each other — "
    "typical for related biomarkers. <b>This is why ElasticNet will outperform Lasso on this dataset.</b> "
    "Lasso would arbitrarily pick one of a correlated pair; ElasticNet shrinks both.")


## A tiny 2-feature companion dataset (for geometry visuals only)

Most of this notebook uses Diabetes (10 features). But you can only **draw** the L1 diamond and L2 circle in 2D — three or more dimensions are impossible to picture on a screen. So we build a tiny synthetic dataset with **just two features** that we'll use later, exclusively for the geometry plots in the Ridge and Lasso sections.

The construction: `y = 3·x₁ + 0.5·x₂ + ε`. Feature 1 is the strong predictor; feature 2 is weak. Regularisation should care a lot about shrinking the wrong-but-tempting wiggles in `x₂` while leaving `x₁` mostly alone.


In [ ]:
# 2-feature toy dataset — used ONLY for geometric visualisations later
rng = np.random.default_rng(RNG)
X_toy = rng.standard_normal((200, 2))
true_beta = np.array([3.0, 0.5])
y_toy = X_toy @ true_beta + rng.standard_normal(200)

print(f"X_toy shape: {X_toy.shape}  (200 rows, 2 features)")
print(f"True coefficients: beta_1 = {true_beta[0]}, beta_2 = {true_beta[1]}")

# OLS fit on toy data — used as the centre of loss contours later
from sklearn.linear_model import LinearRegression as _LR
ols_toy = _LR(fit_intercept=False).fit(X_toy, y_toy)
print(f"OLS estimate:      beta_1 = {ols_toy.coef_[0]:.3f}, beta_2 = {ols_toy.coef_[1]:.3f}")


X_toy shape: (200, 2)  (200 rows, 2 features)
True coefficients: beta_1 = 3.0, beta_2 = 0.5
OLS estimate:      beta_1 = 2.980, beta_2 = 0.449


In [ ]:
box("tip", "Why a separate toy dataset?",
    "The L1 diamond / L2 circle / contour-touching pictures are 2D pictures. On Diabetes with 10 features, the constraint region is a 10-dimensional polytope — you cannot draw it. "
    "For every plot that needs <i>geometric intuition</i>, we'll use this 2-feature toy. For every plot about <i>real predictive performance</i>, we go back to Diabetes.")


In [ ]:
np.random.seed(42)

In [ ]:
np.random.randint(0,10)

3

## Part 1A — The single-split problem (motivating CV)

Before we get to CV, let's see WHY we need it. Train a `LinearRegression` four times, each with a different `random_state` for the train/test split. The test R² changes substantially.


In [ ]:
split_results = []
for rs in [42, 0, 7, 123]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=rs)
    model = LinearRegression().fit(X_tr, y_tr)
    test_r2 = model.score(X_te, y_te)
    split_results.append((rs, test_r2))
    print(f"random_state={rs:>3}: test R² = {test_r2:.3f}")


random_state= 42: test R² = 0.453
random_state=  0: test R² = 0.332
random_state=  7: test R² = 0.404
random_state=123: test R² = 0.568


In [ ]:
rs_vals = [r[0] for r in split_results]
r2_vals = [r[1] for r in split_results]
fig = go.Figure(data=go.Bar(x=[f"rs={r}" for r in rs_vals], y=r2_vals, marker_color=BLUE,
                            text=[f"{v:.3f}" for v in r2_vals], textposition="outside",
                            hovertemplate="random_state %{x}<br>test R² = %{y:.3f}<extra></extra>"))
fig.update_layout(
    title="Same model, same data, 4 different random_states for train/test split",
    yaxis_title="Test R²",
    yaxis=dict(range=[0, 0.6]),
    paper_bgcolor="white", plot_bgcolor="white", width=650, height=400,
)
fig.show()


In [ ]:
spread = max(r2_vals) - min(r2_vals)
mean_r2 = np.mean(r2_vals)
box("danger", "The single test R² is one sample from a noisy distribution",
    f"Across 4 random splits, test R² ranges from {min(r2_vals):.3f} to {max(r2_vals):.3f}. "
    f"Spread = {spread:.3f}, mean = {mean_r2:.3f}. "
    "Reporting a single test R² without uncertainty is dishonest. "
    "Cross-validation gives you a stable estimate AND its uncertainty.")


## Part 1B — K-fold cross-validation, manually

The fix for the single-split problem: instead of doing one train/test split, do **K** of them, where each row plays the role of 'validation' exactly once. This is **K-fold cross-validation**.

Picture the rotation:


In [ ]:
# Visualise the K-fold rotation as a heatmap (5 folds × 10 row-groups)
K = 5
n_groups = 10  # schematic — each 'group' represents a chunk of rows
grid = np.zeros((K, n_groups), dtype=int)
rows_per_fold = n_groups // K
for fold in range(K):
    # fold i holds rows i*rows_per_fold .. (i+1)*rows_per_fold as VAL = 1
    grid[fold, fold*rows_per_fold:(fold+1)*rows_per_fold] = 1

# Build hover labels
role = np.where(grid == 1, "VAL", "TRAIN")

fig = go.Figure(data=go.Heatmap(
    z=grid,
    x=[f"rows {i*10}–{(i+1)*10-1}" for i in range(n_groups)],
    y=[f"Iteration {k+1}" for k in range(K)],
    colorscale=[[0, "#e0e0e0"], [1, MASAI_RED]],
    showscale=False,
    text=role,
    texttemplate="%{text}",
    textfont=dict(color="white", size=11),
    hovertemplate="%{y}<br>%{x}<br>role: %{text}<extra></extra>",
    xgap=2, ygap=4,
))
fig.update_layout(
    title="5-fold cross-validation — each row-group is VAL exactly once",
    paper_bgcolor="white", plot_bgcolor="white",
    width=900, height=380,
    yaxis=dict(autorange="reversed"),
)
fig.show()


In [ ]:
box("definition", "Reading the K-fold rotation",
    "Each row of the diagram is one iteration of cross-validation. The red cells are the validation rows for that iteration; the grey cells are the training rows. "
    "<b>Every row of data is a validation row exactly once.</b> Every iteration trains a fresh model on the grey cells and scores it on the red cells. "
    "At the end, you have K scores — one per iteration. The MEAN is your CV score; the STD is your uncertainty band.<br><br>"
    "With K=5, your estimate is about √5 ≈ 2.2× more stable than a single split. With K=10, about 3.2× more stable.")


### Doing K-fold manually — to understand what `cross_val_score` is hiding

We'll do the rotation the hard way first, so the `cross_val_score` one-liner makes sense afterward. (**Pedagogical pattern: step-by-step before library.**)


In [ ]:
# Manual 5-fold CV
X_full = X.values  # numpy for indexing
y_full = y.values

kf = KFold(n_splits=5, shuffle=True, random_state=RNG)
fold_scores = []
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_full), start=1):
    X_tr, X_va = X_full[train_idx], X_full[val_idx]
    y_tr, y_va = y_full[train_idx], y_full[val_idx]

    model = LinearRegression().fit(X_tr, y_tr)
    score = model.score(X_va, y_va)
    fold_scores.append(score)
    print(f"Fold {fold_idx}: train rows = {len(train_idx)}, val rows = {len(val_idx)}, R² = {score:.3f}")

fold_scores = np.array(fold_scores)
print(f"\nManual CV R² = {fold_scores.mean():.3f} ± {fold_scores.std():.3f}")


Fold 1: train rows = 353, val rows = 89, R² = 0.453
Fold 2: train rows = 353, val rows = 89, R² = 0.573
Fold 3: train rows = 354, val rows = 88, R² = 0.391
Fold 4: train rows = 354, val rows = 88, R² = 0.584
Fold 5: train rows = 354, val rows = 88, R² = 0.391

Manual CV R² = 0.478 ± 0.085


In [ ]:
# Now the one-liner — should give the same result
cv = KFold(n_splits=5, shuffle=True, random_state=RNG)
scores = cross_val_score(LinearRegression(), X_full, y_full, cv=cv, scoring="r2")
print(f"cross_val_score CV R² = {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Fold-by-fold: {[f'{s:.3f}' for s in scores]}")


cross_val_score CV R² = 0.478 ± 0.085
Fold-by-fold: ['0.453', '0.573', '0.391', '0.584', '0.391']


In [ ]:
box("tip", "`cross_val_score` is the same as the manual loop",
    "Same scores, same average. The one-liner is faster to type but doesn't do anything magic. "
    "Knowing the manual version helps when you debug — e.g., 'is my model getting different folds than I think?'")


## Part 1C — The leakage trap

We will now show, on real data, why fitting preprocessing OUTSIDE cross_val_score is biased. Then we fix it with `Pipeline`.


In [ ]:
box("danger", "Why fitting the scaler on full X is leakage",
    "`StandardScaler` computes a mean and standard deviation from its input. If you call `.fit_transform(X)` on the FULL dataset before splitting, the scaler has seen rows that will later be used as validation. "
    "When `cross_val_score` then carves out validation folds, the model trained on each fold has indirectly seen statistics from its own validation rows. The CV score becomes <b>optimistic</b> — it measures generalisation to data the preprocessor has already touched, not to truly new data.<br><br>"
    "On Diabetes (already scaled by sklearn) the leakage is tiny — a few thousandths of an R². On unscaled real-world data with outliers, it can be <b>0.005 to 0.015 R²</b> — small numerically but real. Kaggle teams have lost prize money to this exact bug.")


In [ ]:
# WRONG: scaler fit on full X before CV
scaler_wrong = StandardScaler()
X_scaled_full = scaler_wrong.fit_transform(X_full)
scores_wrong = cross_val_score(LinearRegression(), X_scaled_full, y_full,
                                cv=KFold(n_splits=5, shuffle=True, random_state=RNG), scoring="r2")
print(f"WRONG (leakage)  CV R² = {scores_wrong.mean():.4f}")


WRONG (leakage)  CV R² = 0.4785


In [ ]:
# RIGHT: Pipeline inside cross_val_score
pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
scores_right = cross_val_score(pipe, X_full, y_full,
                                cv=KFold(n_splits=5, shuffle=True, random_state=RNG), scoring="r2")
print(f"RIGHT (no leak)  CV R² = {scores_right.mean():.4f}")


RIGHT (no leak)  CV R² = 0.4785


In [ ]:
diff = scores_wrong.mean() - scores_right.mean()
box("danger", "The leakage gap",
    f"Wrong way: CV R² = {scores_wrong.mean():.4f}<br>"
    f"Right way: CV R² = {scores_right.mean():.4f}<br>"
    f"Difference: {diff:+.4f}<br><br>"
    "On Diabetes (which is already scaled) the leakage is tiny. "
    "On unscaled real-world data the leakage can be several R² points. "
    "Either way: the Pipeline pattern is the rule.")


## Part 1D — Overfitting via polynomial expansion

We deliberately make the model overfit by adding polynomial features. Watch train R² climb while test R² (and CV R²) collapses. **Pedagogical pattern: visual evidence before definition.**

First, we'll **see** overfitting on a tiny 1D toy problem — three side-by-side fits at degree 1, 5, and 15. Then we'll measure it formally on Diabetes.


In [ ]:
# Build a tiny 1D regression problem so we can SEE the polynomial fits
rng_poly = np.random.default_rng(0)
n_poly = 25
x_poly = np.sort(rng_poly.uniform(-3, 3, n_poly))
true_curve = lambda x: 0.5 * x + 0.8 * np.sin(x) + 0.15 * x**2
y_poly = true_curve(x_poly) + rng_poly.normal(0, 0.6, n_poly)

# A held-out test set from the SAME data-generating process
x_te = np.sort(rng_poly.uniform(-3, 3, 50))
y_te = true_curve(x_te) + rng_poly.normal(0, 0.6, 50)

x_smooth = np.linspace(-3, 3, 200)
y_smooth_true = true_curve(x_smooth)

# Fit three polynomials
from numpy.polynomial import Polynomial as _Poly
fits = {}
for d in [1, 5, 15]:
    p = _Poly.fit(x_poly, y_poly, deg=d)
    train_r2 = r2_score(y_poly, p(x_poly))
    test_r2  = r2_score(y_te, p(x_te))
    fits[d] = dict(poly=p, train_r2=train_r2, test_r2=test_r2)
    print(f"degree {d:>2}: train R² = {train_r2:.3f}, test R² = {test_r2:.3f}")


degree  1: train R² = 0.816, test R² = 0.775
degree  5: train R² = 0.944, test R² = 0.800
degree 15: train R² = 0.968, test R² = -97.168


In [ ]:
# Three-panel subplot: same data, three model complexities
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=(
                        f"Degree 1 (line)<br>train R² = {fits[1]['train_r2']:.2f}, test R² = {fits[1]['test_r2']:.2f}",
                        f"Degree 5 (curve)<br>train R² = {fits[5]['train_r2']:.2f}, test R² = {fits[5]['test_r2']:.2f}",
                        f"Degree 15 (wiggle)<br>train R² = {fits[15]['train_r2']:.2f}, test R² = {fits[15]['test_r2']:.2f}",
                    ),
                    horizontal_spacing=0.08)

for i, d in enumerate([1, 5, 15], start=1):
    # true curve (dashed grey)
    fig.add_trace(go.Scatter(x=x_smooth, y=y_smooth_true, mode="lines",
                             line=dict(color="#888", width=2, dash="dash"),
                             name="true curve", showlegend=(i == 1),
                             hovertemplate="true curve<extra></extra>"),
                  row=1, col=i)
    # training data
    fig.add_trace(go.Scatter(x=x_poly, y=y_poly, mode="markers",
                             marker=dict(color=BLUE, size=8, opacity=0.8),
                             name="training data", showlegend=(i == 1),
                             hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>"),
                  row=1, col=i)
    # fitted polynomial
    y_fit_smooth = fits[d]["poly"](x_smooth)
    fig.add_trace(go.Scatter(x=x_smooth, y=y_fit_smooth, mode="lines",
                             line=dict(color=MASAI_RED, width=3),
                             name=f"degree-{d} fit", showlegend=(i == 1),
                             hovertemplate=f"degree {d} fit<br>x=%{{x:.2f}}<br>ŷ=%{{y:.2f}}<extra></extra>"),
                  row=1, col=i)
    fig.update_xaxes(range=[-3.2, 3.2], row=1, col=i)
    fig.update_yaxes(range=[-3.5, 4.5], row=1, col=i)

fig.update_layout(
    title="Same 25 data points, three model complexities",
    paper_bgcolor="white", plot_bgcolor="white",
    width=1000, height=420,
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.5),
)
fig.show()


In [ ]:
box("output", "Reading the three fits",
    f"<b>Degree 1 (underfit)</b>: train R² = {fits[1]['train_r2']:.2f}, test R² = {fits[1]['test_r2']:.2f}. "
    "A straight line misses the curvature on both sides. Train and test are both poor and similar — <b>this is the high-bias quadrant of the dartboard</b>.<br><br>"
    f"<b>Degree 5 (sweet spot)</b>: train R² = {fits[5]['train_r2']:.2f}, test R² = {fits[5]['test_r2']:.2f}. "
    "Curve matches the true shape closely. Train and test are both good and similar.<br><br>"
    f"<b>Degree 15 (overfit)</b>: train R² = {fits[15]['train_r2']:.2f}, test R² = {fits[15]['test_r2']:.2f}. "
    "The polynomial passes through nearly every training point but wiggles wildly between them. Train R² is great, test R² collapses — <b>this is the high-variance quadrant of the dartboard, in code</b>.")


### From the visual to the numbers — sweeping polynomial degree on Diabetes

Now switch back to Diabetes. We'll add polynomial features to the existing 10 features and sweep degree 1..6, computing both training R² and 5-fold CV R² at each. The same U-shape should appear.


In [ ]:
# Sweep polynomial degree 1..6 and measure CV R² + training R²
degrees = range(1, 7)
train_r2_list, cv_r2_list, cv_std_list = [], [], []

for degree in degrees:
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model",  LinearRegression()),
    ])
    pipe.fit(X_full, y_full)
    train_r2 = pipe.score(X_full, y_full)

    cv_scores = cross_val_score(pipe, X_full, y_full,
                                 cv=KFold(n_splits=5, shuffle=True, random_state=RNG), scoring="r2")
    train_r2_list.append(train_r2)
    cv_r2_list.append(cv_scores.mean())
    cv_std_list.append(cv_scores.std())
    print(f"degree={degree}: train R² = {train_r2:.3f}, CV R² = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


degree=1: train R² = 0.518, CV R² = 0.478 ± 0.085
degree=2: train R² = 0.592, CV R² = 0.366 ± 0.106
degree=3: train R² = 0.798, CV R² = -42.933 ± 36.172
degree=4: train R² = 1.000, CV R² = -32.450 ± 9.913
degree=5: train R² = 1.000, CV R² = -11.833 ± 2.207
degree=6: train R² = 1.000, CV R² = -17.468 ± 9.634


In [ ]:
# Plot the U-shape: train vs CV error
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(degrees), y=train_r2_list,
    mode="lines+markers", name="Training R²", line=dict(color=MASAI_RED, width=3),
    hovertemplate="degree %{x}<br>train R² = %{y:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=list(degrees), y=cv_r2_list,
    mode="lines+markers", name="CV R² (test estimate)", line=dict(color=BLUE, width=3),
    error_y=dict(type="data", array=cv_std_list, visible=True),
    hovertemplate="degree %{x}<br>CV R² = %{y:.3f}<extra></extra>",
))
fig.update_layout(
    title="The U-shape: training R² rises monotonically, CV R² peaks and falls",
    xaxis_title="Polynomial degree (= model complexity)",
    yaxis_title="R²",
    paper_bgcolor="white", plot_bgcolor="white",
    width=750, height=450,
    yaxis=dict(gridcolor="#eee"),
)
fig.show()


In [ ]:
sweet = int(np.argmax(cv_r2_list)) + 1
box("output", "Reading the U-shape",
    f"Training R² climbs monotonically — at degree=6 the model can fit training data near-perfectly. "
    f"CV R² peaks at degree={sweet} (best CV R² = {max(cv_r2_list):.3f}) and then collapses. "
    "The peak is the bias-variance sweet spot. Past it, training fit keeps improving but the model is just memorising noise.")


## Bias, Variance, Irreducible Error — the dartboard mental model

We've now SEEN the U-shape empirically. Time to name what we're seeing. The classic mental model is a **dartboard**.

Imagine you train the same model on many *different samples* of data. Each trained model is one dart. The TRUE answer is the bullseye. The pattern of darts tells you about your model's bias and variance.


In [ ]:
# Simulate 4 different 'shooters' — each one is a (bias, variance) regime
rng_dart = np.random.default_rng(7)
n_darts = 40

# (mean_offset_x, mean_offset_y, scatter_std)
scenarios = {
    "Low bias, low variance<br>(the goal)":            (0.0, 0.0, 0.18),
    "High bias, low variance<br>(systematically wrong)": (1.2, 0.9, 0.18),
    "Low bias, high variance<br>(right on average)":     (0.0, 0.0, 0.65),
    "High bias, high variance<br>(broken)":              (1.2, 0.9, 0.65),
}

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=list(scenarios.keys()),
                    horizontal_spacing=0.15, vertical_spacing=0.18)

for idx, (label, (mx, my, sd)) in enumerate(scenarios.items()):
    row = idx // 2 + 1
    col = idx % 2 + 1
    darts_x = rng_dart.normal(mx, sd, n_darts)
    darts_y = rng_dart.normal(my, sd, n_darts)
    # concentric rings
    for r, ring_color in [(0.2, "#fff3e0"), (0.5, "#ffe0b2"), (0.9, "#fce4ec")]:
        theta = np.linspace(0, 2*np.pi, 80)
        fig.add_trace(go.Scatter(x=r*np.cos(theta), y=r*np.sin(theta),
                                 mode="lines", line=dict(color="#bbb", width=1),
                                 showlegend=False, hoverinfo="skip"),
                      row=row, col=col)
    # bullseye marker
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers",
                             marker=dict(color=MASAI_RED, size=14, symbol="x"),
                             name="bullseye (truth)", showlegend=(idx == 0),
                             hovertemplate="truth (0, 0)<extra></extra>"),
                  row=row, col=col)
    # darts
    dist_from_truth = np.sqrt(darts_x**2 + darts_y**2)
    fig.add_trace(go.Scatter(x=darts_x, y=darts_y, mode="markers",
                             marker=dict(color=BLUE, size=7, opacity=0.7),
                             name="trained models", showlegend=(idx == 0),
                             customdata=dist_from_truth,
                             hovertemplate="dart at (%{x:.2f}, %{y:.2f})<br>distance from truth = %{customdata:.2f}<extra></extra>"),
                  row=row, col=col)

for r in range(1, 3):
    for c in range(1, 3):
        fig.update_xaxes(range=[-2, 2.5], showgrid=False, zeroline=False, row=r, col=c, scaleanchor=f"y{r*2-2+c if (r,c)!=(1,1) else ''}", scaleratio=1)
        fig.update_yaxes(range=[-2, 2.5], showgrid=False, zeroline=False, row=r, col=c)

fig.update_layout(
    title="The bias-variance dartboard — each dart is one trained model",
    paper_bgcolor="white", plot_bgcolor="white",
    width=850, height=750,
)
fig.show()


In [ ]:
box("output", "Reading the dartboard",
    "<b>Low bias, low variance (top-left):</b> every model lands near the truth AND near each other. This is the goal.<br>"
    "<b>High bias, low variance (top-right):</b> tight cluster, but systematically off. The model is consistently wrong — usually because it's too simple (e.g., a line on curved data). Symptom: train R² and test R² are both bad and similar.<br>"
    "<b>Low bias, high variance (bottom-left):</b> centred on truth but each model is far from the next. <b>Right on average, individually unreliable.</b> Usually because the model is too flexible. Symptom: train R² is near 1.0 but test R² collapses (overfitting).<br>"
    "<b>High bias, high variance (bottom-right):</b> wrong AND unreliable. Time to throw the model out.")


In [ ]:
box("definition", "Bias — formally",
    "Bias is the <b>systematic part</b> of the error. If you trained your model on infinite different datasets and averaged the predictions, bias is how far that average is from the truth.<br><br>"
    "Sources of bias: model is too simple for the underlying pattern. <i>Fitting a line to clearly curved data is the canonical example.</i> No amount of more data fixes bias — only changing the model class does.")


In [ ]:
box("definition", "Variance — formally",
    "Variance is how much the model's predictions <b>change when the training data changes</b>. A high-variance model, retrained on a slightly different sample, produces a very different model.<br><br>"
    "Sources of variance: model is too flexible. <i>A 50-leaf decision tree on 100 data points is the canonical example</i> — change one point, the tree reshapes. High variance is what 'overfitting' looks like in slow motion.")


In [ ]:
box("math", "The decomposition formula",
    "For any model trained on a random sample, the expected squared error on a new point decomposes:<br>"
    "<b>E[(y − ŷ)²] = Bias² + Variance + σ²</b><br><br>"
    "Three numbers; only two are under your control. <b>σ² (irreducible error)</b> is the noise floor — measurement error in the data itself. Knowing it exists prevents you from chasing zero error.<br><br>"
    "<i>We don't derive this today. The key takeaway: bias and variance trade off. Most things that lower one raise the other.</i>")


In [ ]:
box("tip", "Why the U-shape exists (the bridge to regularisation)",
    "As model complexity rises (more features, deeper trees, higher polynomial degree):<br>"
    "• <b>Bias falls</b> — the model can finally express the true pattern.<br>"
    "• <b>Variance rises</b> — the model also starts modelling noise as if it were signal.<br><br>"
    "Bias² + Variance is a sum where one term is falling and the other is rising → there's a minimum somewhere. <b>That minimum is the sweet spot of the U-shape.</b><br><br>"
    "<b>Regularisation</b>, the second half of this notebook, is one way to slide LEFTWARD on the U from the overfitting side — take a complex model and constrain it back toward the sweet spot.")


## Part 1 summary

- A single train/test split's R² is noisy. **Use k-fold cross-validation** to get a stable estimate.
- For regression, `KFold(n_splits=5, shuffle=True, random_state=42)` is the canonical default.
- Preprocessing that involves `.fit()` MUST be inside a `Pipeline` evaluated by `cross_val_score`. Otherwise: leakage.
- High-complexity models overfit. The training R² climbs to 1.0; the CV R² collapses. Diagnose by plotting both vs complexity.

**Next:** instead of reducing complexity (fewer features, lower polynomial degree), we *keep* the complex model and add a **penalty** on coefficient size. That's regularisation. The art is in the penalty's shape.


## Part 2A — Setup for regularisation

Hold out 20% as the final test set. **Touch it once at the end** — never during alpha tuning.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RNG)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Baseline OLS CV score on the training set
ols_pipe = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
ols_cv = cross_val_score(ols_pipe, X_train, y_train,
                          cv=KFold(n_splits=5, shuffle=True, random_state=RNG), scoring="r2")
print(f"\nOLS CV R² on TRAIN: {ols_cv.mean():.3f} ± {ols_cv.std():.3f}")


Train: (353, 10), Test: (89, 10)

OLS CV R² on TRAIN: 0.480 ± 0.041


## Part 2B — Ridge regression (L2)

Ridge minimises `||y − Xβ||² + α · ||β||₂²`. The L2 penalty grows quadratically with coefficient size. Geometrically: the constraint region is a CIRCLE — smooth, no corners.


In [ ]:
box("math", "Ridge objective",
    "<b>min_β  ||y − Xβ||² + α · (β₁² + β₂² + … + βₚ²)</b><br>"
    "α is the regularisation strength. α → 0 recovers OLS. α → ∞ pushes all coefficients to zero.<br>"
    "Geometry: constraint region is a circle (in 2D) — touches the loss contour smoothly, NOT at a corner.")


### The L2 geometry — circle constraint touching loss contours

Time to SEE why Ridge shrinks but doesn't zero out. We use the 2-feature toy dataset built earlier (`X_toy`, `y_toy`) because two dimensions is the only space where we can actually draw the picture.

The picture has three ingredients:

1. **Loss contours** — sets of `(β₁, β₂)` with the same training error. They are ellipses centred on the OLS solution (the unconstrained best fit).
2. **The L2 ball** — a circle of some radius `r` centred on the origin. Anything inside the circle satisfies `β₁² + β₂² ≤ r²`. The larger α is, the smaller `r` is.
3. **The Ridge solution** — the point where the smallest loss contour that still touches the L2 ball meets the ball.


In [ ]:
# Build the loss surface on a grid of (beta_1, beta_2) values
b1 = np.linspace(-1.0, 4.0, 220)
b2 = np.linspace(-2.5, 2.5, 220)
B1, B2 = np.meshgrid(b1, b2)

# Residual sum of squares at each (beta_1, beta_2)
def rss(b1g, b2g):
    pred = X_toy[:, 0:1] * b1g.ravel() + X_toy[:, 1:2] * b2g.ravel()
    res = y_toy[:, None] - pred
    return (res**2).sum(axis=0).reshape(b1g.shape)

RSS = rss(B1, B2)
ols_b1, ols_b2 = ols_toy.coef_
rss_at_ols = ((y_toy - X_toy @ ols_toy.coef_)**2).sum()

# Pick a few contour levels above the OLS minimum
contour_levels = rss_at_ols + np.array([10, 40, 100, 200, 400])
print(f"OLS optimum: (beta_1, beta_2) = ({ols_b1:.3f}, {ols_b2:.3f})")


OLS optimum: (beta_1, beta_2) = (2.980, 0.449)


In [ ]:
# Fit Ridge at alpha=10 on the toy data to find the L2 contact point
ridge_toy = Ridge(alpha=10.0, fit_intercept=False).fit(X_toy, y_toy)
ridge_b1, ridge_b2 = ridge_toy.coef_
l2_radius = np.sqrt(ridge_b1**2 + ridge_b2**2)
print(f"Ridge solution (alpha=10): (beta_1, beta_2) = ({ridge_b1:.3f}, {ridge_b2:.3f})")
print(f"L2 ball radius at this alpha: r = sqrt(beta_1² + beta_2²) = {l2_radius:.3f}")

# Build the circle for plotting
theta = np.linspace(0, 2*np.pi, 200)
circle_x = l2_radius * np.cos(theta)
circle_y = l2_radius * np.sin(theta)


Ridge solution (alpha=10): (beta_1, beta_2) = (2.836, 0.433)
L2 ball radius at this alpha: r = sqrt(beta_1² + beta_2²) = 2.869


In [ ]:
# Plot loss contours + L2 circle + OLS + Ridge solutions
fig = go.Figure()

# Loss contours
fig.add_trace(go.Contour(
    z=RSS, x=b1, y=b2,
    contours=dict(start=contour_levels[0], end=contour_levels[-1],
                  size=(contour_levels[-1]-contour_levels[0])/4,
                  coloring="lines", showlabels=False),
    line=dict(width=1.5, color="#999"),
    showscale=False,
    hovertemplate="β₁=%{x:.2f}, β₂=%{y:.2f}<br>loss=%{z:.1f}<extra></extra>",
    name="loss contours",
))

# L2 ball (circle)
fig.add_trace(go.Scatter(
    x=circle_x, y=circle_y, mode="lines",
    line=dict(color=BLUE, width=3),
    name=f"L2 ball (r={l2_radius:.2f})",
    fill="toself", fillcolor="rgba(21,101,192,0.08)",
    hovertemplate="on the L2 ball<br>β₁²+β₂² = r²<extra></extra>",
))

# OLS optimum (unconstrained)
fig.add_trace(go.Scatter(
    x=[ols_b1], y=[ols_b2], mode="markers+text",
    marker=dict(color="#888", size=14, symbol="star"),
    text=["OLS"], textposition="top center",
    name="OLS optimum (unconstrained)",
    hovertemplate=f"OLS<br>β₁={ols_b1:.2f}, β₂={ols_b2:.2f}<extra></extra>",
))

# Ridge contact point
fig.add_trace(go.Scatter(
    x=[ridge_b1], y=[ridge_b2], mode="markers+text",
    marker=dict(color=MASAI_RED, size=14, symbol="diamond"),
    text=["Ridge"], textposition="bottom right",
    name="Ridge solution",
    hovertemplate=f"Ridge (α=10)<br>β₁={ridge_b1:.2f}, β₂={ridge_b2:.2f}<br><b>both non-zero</b><extra></extra>",
))

# Axes
fig.add_hline(y=0, line_color="#ccc", line_width=1)
fig.add_vline(x=0, line_color="#ccc", line_width=1)

fig.update_layout(
    title="L2 geometry — circle constraint, smooth contact, no zeros",
    xaxis_title="β₁", yaxis_title="β₂",
    paper_bgcolor="white", plot_bgcolor="white",
    width=720, height=620,
    xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1, 4]),
    yaxis=dict(range=[-2.5, 2.5]),
)
fig.show()


In [ ]:
box("definition", "Reading the L2 geometry",
    f"The grey ellipses are <b>loss contours</b> — sets of (β₁, β₂) with the same training error. They circle around the OLS optimum at ({ols_b1:.2f}, {ols_b2:.2f}).<br><br>"
    f"The blue circle is the <b>L2 ball</b> at this alpha. Its radius is r = {l2_radius:.2f}.<br><br>"
    f"The red diamond is the <b>Ridge solution</b> — the point on the circle where the smallest loss contour first touches it. "
    "<b>Crucially, the contact point is on the smooth arc of the circle.</b> A circle has no corners. The contact happens somewhere on its surface where the contour ellipse meets it tangentially. "
    f"At that contact point, both coordinates are non-zero (β₁={ridge_b1:.2f}, β₂={ridge_b2:.2f}). <b>Ridge shrinks; it never eliminates.</b>")


In [ ]:
# Single Ridge fit at alpha=1
ridge_pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0, random_state=RNG))])
ridge_pipe.fit(X_train, y_train)
ridge_coefs = ridge_pipe.named_steps["ridge"].coef_
print(f"Ridge coefficients (alpha=1): {dict(zip(feature_names, ridge_coefs.round(2)))}")
print(f"\nNumber of non-zero coefficients: {sum(ridge_coefs != 0)} / {len(ridge_coefs)}")


Ridge coefficients (alpha=1): {'age': np.float64(1.81), 'sex': np.float64(-11.45), 'bmi': np.float64(25.73), 'bp': np.float64(16.73), 's1': np.float64(-34.67), 's2': np.float64(17.05), 's3': np.float64(3.37), 's4': np.float64(11.76), 's5': np.float64(31.38), 's6': np.float64(2.46)}

Number of non-zero coefficients: 10 / 10


### Ridge coefficient path — sweep alpha

We fit Ridge at many alphas and plot how each coefficient evolves. **Pedagogical pattern: signature visualization.**


In [ ]:
alphas_ridge = np.logspace(-2, 4, 60)
ridge_path = []
for alpha in alphas_ridge:
    pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=alpha))])
    pipe.fit(X_train, y_train)
    ridge_path.append(pipe.named_steps["ridge"].coef_)
ridge_path = np.array(ridge_path)
print(f"Path shape: {ridge_path.shape}  (alphas × features)")


Path shape: (60, 10)  (alphas × features)


In [ ]:
# Plot all 10 coefficient paths
fig = go.Figure()
for i, feat in enumerate(feature_names):
    fig.add_trace(go.Scatter(
        x=alphas_ridge, y=ridge_path[:, i],
        mode="lines", name=feat, line=dict(width=2),
        hovertemplate=f"<b>{feat}</b><br>alpha = %{{x:.3f}}<br>coef = %{{y:.2f}}<extra></extra>",
    ))
fig.update_layout(
    title="Ridge coefficient path on Diabetes — smooth shrinkage, never zero",
    xaxis_title="alpha (log scale)",
    yaxis_title="coefficient value",
    xaxis_type="log",
    paper_bgcolor="white", plot_bgcolor="white",
    width=850, height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()


In [ ]:
nonzero_at_max = sum(np.abs(ridge_path[-1]) > 1e-6)
box("output", "Ridge coefficient path — what to see",
    f"At the largest alpha tested ({alphas_ridge[-1]:.0f}), {nonzero_at_max} of {len(feature_names)} coefficients are STILL non-zero. "
    "All paths smoothly drift toward zero but none actually reach it. "
    "This is Ridge's signature: shrink everything, eliminate nothing.")


## Part 2C — Lasso regression (L1) — THE sparsity moment

Lasso minimises `||y − Xβ||² + α · ||β||₁`. The L1 penalty grows linearly with coefficient magnitude. Geometrically: the constraint region is a DIAMOND — corners ON the coordinate axes.


In [ ]:
box("math", "Lasso objective",
    "<b>min_β  ||y − Xβ||² + α · (|β₁| + |β₂| + … + |βₚ|)</b><br>"
    "Geometry: constraint region is a diamond with corners on the axes. "
    "<b>When the loss contour touches the diamond at a corner, one coordinate of β is exactly zero.</b> "
    "That's SPARSITY — baked into the geometry.")


### The L1 geometry — diamond corners and the sparsity argument

This is the most important conceptual moment in the regularisation half of the notebook. **Why does Lasso produce exact zeros while Ridge doesn't?**

Same picture as Ridge, but the constraint region is **a diamond** instead of a circle. The diamond's corners lie *on the coordinate axes*. When the loss contour shrinks until it just touches the diamond, the contact point is overwhelmingly likely to be **at a corner** — not on an edge.

And at a corner, one coordinate of β is exactly zero. That feature has been *eliminated* from the model. That's sparsity, and it's built into the shape of the constraint, not into the algorithm.


In [ ]:
# Fit Lasso on the toy data — pick alpha so the contact lands at a corner
lasso_toy = Lasso(alpha=0.3, fit_intercept=False, max_iter=50_000).fit(X_toy, y_toy)
lasso_b1, lasso_b2 = lasso_toy.coef_
l1_budget = abs(lasso_b1) + abs(lasso_b2)
print(f"Lasso solution (alpha=0.3): (beta_1, beta_2) = ({lasso_b1:.3f}, {lasso_b2:.3f})")
print(f"L1 budget = |β₁| + |β₂| = {l1_budget:.3f}")
if abs(lasso_b2) < 1e-6:
    print("--> β₂ = 0 EXACTLY. Feature 2 has been eliminated.")


Lasso solution (alpha=0.3): (beta_1, beta_2) = (2.691, 0.109)
L1 budget = |β₁| + |β₂| = 2.800


In [ ]:
# Build the diamond constraint region
# Vertices: (±L1_budget, 0) and (0, ±L1_budget)
diamond_x = [l1_budget, 0, -l1_budget, 0, l1_budget]
diamond_y = [0, l1_budget, 0, -l1_budget, 0]

fig = go.Figure()

# Loss contours (same RSS surface as the L2 plot — uses X_toy, y_toy)
fig.add_trace(go.Contour(
    z=RSS, x=b1, y=b2,
    contours=dict(start=contour_levels[0], end=contour_levels[-1],
                  size=(contour_levels[-1]-contour_levels[0])/4,
                  coloring="lines", showlabels=False),
    line=dict(width=1.5, color="#999"),
    showscale=False,
    hovertemplate="β₁=%{x:.2f}, β₂=%{y:.2f}<br>loss=%{z:.1f}<extra></extra>",
    name="loss contours",
))

# Diamond constraint
fig.add_trace(go.Scatter(
    x=diamond_x, y=diamond_y, mode="lines",
    line=dict(color=MASAI_RED, width=3),
    fill="toself", fillcolor="rgba(237,28,36,0.08)",
    name=f"L1 ball (|β₁|+|β₂|={l1_budget:.2f})",
    hovertemplate="on the L1 diamond<br>|β₁|+|β₂| = budget<extra></extra>",
))

# Highlight the four corners — these are where one coordinate is exactly 0
corner_x = [l1_budget, 0, -l1_budget, 0]
corner_y = [0, l1_budget, 0, -l1_budget]
corner_labels = [
    f"corner (β₁={l1_budget:.2f}, β₂=0)",
    f"corner (β₁=0, β₂={l1_budget:.2f})",
    f"corner (β₁=-{l1_budget:.2f}, β₂=0)",
    f"corner (β₁=0, β₂=-{l1_budget:.2f})",
]
fig.add_trace(go.Scatter(
    x=corner_x, y=corner_y, mode="markers",
    marker=dict(color=MASAI_RED, size=11, symbol="square-open", line=dict(width=2)),
    name="diamond corners (on axes)",
    text=corner_labels,
    hovertemplate="%{text}<br><b>one coordinate is exactly zero</b><extra></extra>",
))

# OLS optimum (unconstrained)
fig.add_trace(go.Scatter(
    x=[ols_b1], y=[ols_b2], mode="markers+text",
    marker=dict(color="#888", size=14, symbol="star"),
    text=["OLS"], textposition="top center",
    name="OLS optimum (unconstrained)",
    hovertemplate=f"OLS<br>β₁={ols_b1:.2f}, β₂={ols_b2:.2f}<extra></extra>",
))

# Lasso contact point
fig.add_trace(go.Scatter(
    x=[lasso_b1], y=[lasso_b2], mode="markers+text",
    marker=dict(color=MASAI_RED, size=18, symbol="diamond", line=dict(color="black", width=2)),
    text=["Lasso"], textposition="top right",
    name="Lasso solution (at a corner!)",
    hovertemplate=f"Lasso (α=0.3)<br>β₁={lasso_b1:.2f}, β₂={lasso_b2:.2f}<br><b>β₂ = 0 — sparsity!</b><extra></extra>",
))

fig.add_hline(y=0, line_color="#ccc", line_width=1)
fig.add_vline(x=0, line_color="#ccc", line_width=1)

fig.update_layout(
    title="L1 geometry — diamond corners ON the axes, contact at a corner → β₂ = 0",
    xaxis_title="β₁", yaxis_title="β₂",
    paper_bgcolor="white", plot_bgcolor="white",
    width=720, height=620,
    xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1, 4]),
    yaxis=dict(range=[-2.5, 2.5]),
)
fig.show()


In [ ]:
box("definition", "Why L1 produces sparsity (the corner argument)",
    "The L1 ball is a <b>diamond</b> — a square rotated 45°. Its corners lie ON the coordinate axes.<br><br>"
    "When you shrink an ellipse-shaped loss contour until it just touches the diamond, the contact happens at a corner <i>far more often than on a flat edge</i>. Intuitively: the corners stick OUT toward the OLS optimum, so they're what an incoming ellipse touches first. The edges are recessed.<br><br>"
    f"<b>At a corner, one or more coordinates of β are exactly zero.</b> In our toy fit, the contact landed at the corner (β₁={lasso_b1:.2f}, β₂=0) — Lasso eliminated feature 2.<br><br>"
    "<b>Contrast with Ridge:</b> the L2 ball is a circle. A circle has no corners. The contact happens somewhere on its smooth arc, and the coordinates are generally both non-zero. <i>Ridge shrinks but never eliminates. Lasso shrinks AND eliminates.</i>")


In [ ]:
box("industry", "Why the geometry argument matters in practice",
    "In high dimensions, the L1 'diamond' becomes a cross-polytope — corners on every axis, edges and faces between them. The same probability intuition holds: the corners point toward the OLS optimum, edges are recessed. <b>Lasso picks corners, and corners are where features get eliminated.</b><br><br>"
    "This is the difference between an algorithm that 'does feature selection because of a special rule' and one that 'does feature selection because of the shape of its constraint region'. The latter is what Lasso is. Once you understand the shape, you don't need to memorise the behaviour.")


In [ ]:
# Single Lasso fit at alpha=1
lasso_pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=1.0, max_iter=10_000, random_state=RNG))])
lasso_pipe.fit(X_train, y_train)
lasso_coefs = lasso_pipe.named_steps["lasso"].coef_
n_nonzero = sum(lasso_coefs != 0)
print(f"Lasso coefficients (alpha=1): {dict(zip(feature_names, lasso_coefs.round(2)))}")
print(f"\nNumber of non-zero coefficients: {n_nonzero} / {len(lasso_coefs)}")
print(f"Surviving features: {[f for f, c in zip(feature_names, lasso_coefs) if c != 0]}")


Lasso coefficients (alpha=1): {'age': np.float64(0.69), 'sex': np.float64(-9.3), 'bmi': np.float64(26.22), 'bp': np.float64(15.66), 's1': np.float64(-8.23), 's2': np.float64(-0.0), 's3': np.float64(-9.02), 's4': np.float64(3.42), 's5': np.float64(22.64), 's6': np.float64(2.1)}

Number of non-zero coefficients: 9 / 10
Surviving features: ['age', 'sex', 'bmi', 'bp', 's1', 's3', 's4', 's5', 's6']


### Lasso coefficient path — sweep alpha

Same chart pattern. Watch coefficients drop to **exactly zero**, one by one, as alpha climbs.


In [ ]:
alphas_lasso = np.logspace(-3, 1, 60)
lasso_path = []
for alpha in alphas_lasso:
    pipe = Pipeline([("scaler", StandardScaler()), ("lasso", Lasso(alpha=alpha, max_iter=10_000, random_state=RNG))])
    pipe.fit(X_train, y_train)
    lasso_path.append(pipe.named_steps["lasso"].coef_)
lasso_path = np.array(lasso_path)

fig = go.Figure()
for i, feat in enumerate(feature_names):
    fig.add_trace(go.Scatter(
        x=alphas_lasso, y=lasso_path[:, i],
        mode="lines", name=feat, line=dict(width=2),
        hovertemplate=f"<b>{feat}</b><br>alpha = %{{x:.3f}}<br>coef = %{{y:.2f}}<extra></extra>",
    ))
fig.update_layout(
    title="Lasso coefficient path — features die abruptly at exactly zero",
    xaxis_title="alpha (log scale)",
    yaxis_title="coefficient value",
    xaxis_type="log",
    paper_bgcolor="white", plot_bgcolor="white",
    width=850, height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()


In [ ]:
# Count non-zero coefficients at each alpha
nonzero_counts = (np.abs(lasso_path) > 1e-6).sum(axis=1)
fig = go.Figure(data=go.Scatter(
    x=alphas_lasso, y=nonzero_counts,
    mode="lines+markers", line=dict(color=MASAI_RED, width=3),
    hovertemplate="alpha = %{x:.3f}<br>non-zero features = %{y}<extra></extra>",
))
fig.update_layout(
    title="Number of non-zero Lasso coefficients vs alpha",
    xaxis_title="alpha (log scale)",
    yaxis_title="# non-zero coefficients",
    xaxis_type="log",
    paper_bgcolor="white", plot_bgcolor="white",
    width=750, height=400,
)
fig.show()


In [ ]:
# Find the alpha that keeps exactly 5 features
idx_5 = np.argmin(np.abs(nonzero_counts - 5))
alpha_5 = alphas_lasso[idx_5]
surviving_5 = [f for f, c in zip(feature_names, lasso_path[idx_5]) if abs(c) > 1e-6]
box("output", "The alpha where ~5 features survive",
    f"At alpha ≈ {alpha_5:.3f}, Lasso keeps {nonzero_counts[idx_5]} features non-zero.<br>"
    f"Surviving features (ordered by appearance): {surviving_5}")


### Ridge vs Lasso paths — side-by-side

Now put the two coefficient paths next to each other on a single figure with a shared y-axis. **Pedagogical pattern: side-by-side comparison.** This is the visual proof that the L2 geometry (circle, no corners) and the L1 geometry (diamond, corners on axes) produce dramatically different shrinkage behaviour.


In [ ]:
# Shared subplot: Ridge path on the left, Lasso path on the right
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=(
                        "Ridge (L2) — smooth shrinkage, never zero",
                        "Lasso (L1) — abrupt drops to exactly zero",
                    ),
                    horizontal_spacing=0.06)

# Generate a colour for each feature, consistent across both panels
feature_colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set2
feature_colors = feature_colors[:len(feature_names)]

# Left: Ridge path
for i, feat in enumerate(feature_names):
    fig.add_trace(go.Scatter(
        x=alphas_ridge, y=ridge_path[:, i],
        mode="lines", name=feat, legendgroup=feat,
        line=dict(color=feature_colors[i], width=2),
        hovertemplate=f"<b>{feat}</b> (Ridge)<br>alpha=%{{x:.3f}}<br>coef=%{{y:.2f}}<extra></extra>",
        showlegend=True,
    ), row=1, col=1)

# Right: Lasso path
for i, feat in enumerate(feature_names):
    fig.add_trace(go.Scatter(
        x=alphas_lasso, y=lasso_path[:, i],
        mode="lines", name=feat, legendgroup=feat, showlegend=False,
        line=dict(color=feature_colors[i], width=2),
        hovertemplate=f"<b>{feat}</b> (Lasso)<br>alpha=%{{x:.3f}}<br>coef=%{{y:.2f}}<extra></extra>",
    ), row=1, col=2)

# Zero reference lines
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="grey", row=1, col=2)

fig.update_xaxes(type="log", title_text="alpha (log)", row=1, col=1)
fig.update_xaxes(type="log", title_text="alpha (log)", row=1, col=2)
fig.update_yaxes(title_text="coefficient value", row=1, col=1)

fig.update_layout(
    title="Ridge vs Lasso coefficient paths on Diabetes — the geometry comes through in the curves",
    paper_bgcolor="white", plot_bgcolor="white",
    width=1100, height=520,
)
fig.show()


In [ ]:
ridge_nonzero_final = sum(np.abs(ridge_path[-1]) > 1e-6)
lasso_nonzero_final = sum(np.abs(lasso_path[-1]) > 1e-6)
box("output", "What the side-by-side reveals",
    f"<b>Ridge (left)</b>: at the largest alpha tested, {ridge_nonzero_final}/{len(feature_names)} coefficients are still non-zero. "
    "Each curve smoothly bends toward zero but never crosses it — exactly what the L2 circle geometry predicts.<br><br>"
    f"<b>Lasso (right)</b>: at the largest alpha tested, only {lasso_nonzero_final}/{len(feature_names)} coefficients remain. "
    "Each curve hits zero abruptly and stays there — exactly what the L1 corner geometry predicts.<br><br>"
    "The mathematical penalty differs by one tiny exponent (β² vs |β|), but the consequence — a circle versus a diamond — is huge in practice.")


## Part 2D — Side-by-side: OLS vs Ridge vs Lasso coefficients

Same data, same train/test split. **Pedagogical pattern: side-by-side comparison.**


In [ ]:
ols = Pipeline([("scaler", StandardScaler()), ("m", LinearRegression())]).fit(X_train, y_train)
ridge_fixed = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=10.0))]).fit(X_train, y_train)
lasso_fixed = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=1.0, max_iter=10_000))]).fit(X_train, y_train)

coefs_df = pd.DataFrame({
    "feature": feature_names,
    "OLS":   ols.named_steps["m"].coef_,
    "Ridge": ridge_fixed.named_steps["m"].coef_,
    "Lasso": lasso_fixed.named_steps["m"].coef_,
})
coefs_df


,feature,OLS,Ridge,Lasso
0,age,1.753758,1.931332,0.687032
1,sex,-11.511809,-11.028192,-9.297519
2,bmi,25.607121,25.605561,26.219225
3,bp,16.828872,16.304191,15.657314
4,s1,-44.448856,-12.934236,-8.228172
5,s2,24.640954,0.538173,-0.000000
6,s3,7.676978,-6.125109,-9.024087
7,s4,13.138784,8.569484,3.420861
8,s5,35.161195,22.749308,22.636465
9,s6,2.351364,2.918146,2.098647


In [ ]:
# Bar chart
fig = go.Figure()
fig.add_trace(go.Bar(x=coefs_df["feature"], y=coefs_df["OLS"],   name="OLS",   marker_color=BLUE))
fig.add_trace(go.Bar(x=coefs_df["feature"], y=coefs_df["Ridge"], name="Ridge (α=10)",  marker_color=GREEN))
fig.add_trace(go.Bar(x=coefs_df["feature"], y=coefs_df["Lasso"], name="Lasso (α=1)",   marker_color=MASAI_RED))
fig.update_layout(
    title="Coefficient comparison — OLS, Ridge, Lasso on Diabetes",
    barmode="group",
    yaxis_title="coefficient",
    paper_bgcolor="white", plot_bgcolor="white",
    width=900, height=450,
)
fig.show()


In [ ]:
lasso_zero = sum(coefs_df["Lasso"] == 0)
ridge_zero = sum(coefs_df["Ridge"] == 0)
box("output", "What the bar chart shows",
    f"<b>OLS</b>: large coefficients, all non-zero ({sum(coefs_df['OLS'] != 0)} of 10).<br>"
    f"<b>Ridge</b>: shrunk toward zero, still all non-zero ({sum(coefs_df['Ridge'] != 0)} of 10, {ridge_zero} exact zeros).<br>"
    f"<b>Lasso</b>: some shrunk, <b>{lasso_zero} exact zeros</b>. Sparsity in action.")


## Part 2E — ElasticNet (L1 + L2 mix)

When you have correlated features (like Diabetes's s1-s6 serums), Lasso arbitrarily picks one and zeros the others. ElasticNet shrinks them together — less sparse, more stable.


In [ ]:
box("definition", "ElasticNet — combined penalty",
    "<b>min_β  ||y − Xβ||² + α [ρ·||β||₁ + (1−ρ)·||β||₂²]</b><br>"
    "<b>l1_ratio = ρ</b> controls the mix:<br>"
    "• l1_ratio = 1 → pure Lasso<br>"
    "• l1_ratio = 0 → pure Ridge<br>"
    "• l1_ratio = 0.5 → equal mix")


### ElasticNet geometry — the constraint region morphs from diamond to circle

What does the ElasticNet ball look like? It's a **rounded diamond** — sharp corners get blunted as you blend in more L2. Watch the shape morph as `l1_ratio` slides from 1 (pure Lasso, sharp corners) through 0.5 (rounded) to 0 (pure Ridge, perfect circle).


In [ ]:
# Build the ElasticNet ball: {(b1, b2) : rho*(|b1|+|b2|) + (1-rho)*(b1²+b2²) <= budget}
# For visualisation, pick a fixed budget for the inequality.
theta = np.linspace(0, 2*np.pi, 400)
budget = 1.0

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=(
                        "l1_ratio = 1.0<br>pure Lasso (diamond)",
                        "l1_ratio = 0.5<br>ElasticNet (rounded diamond)",
                        "l1_ratio = 0.0<br>pure Ridge (circle)",
                    ),
                    horizontal_spacing=0.08)

for col, rho in enumerate([1.0, 0.5, 0.0], start=1):
    # For each direction theta, find the radius r where the penalty hits the budget
    # Penalty(r, theta) = rho*r*(|cos|+|sin|) + (1-rho)*r²
    rs = np.zeros_like(theta)
    for i, t in enumerate(theta):
        c, s = abs(np.cos(t)), abs(np.sin(t))
        a = (1 - rho)
        b = rho * (c + s)
        d = -budget
        if a < 1e-12:
            rs[i] = budget / b if b > 0 else 0
        else:
            disc = b*b - 4*a*d
            rs[i] = (-b + np.sqrt(disc)) / (2*a)
    xs = rs * np.cos(theta)
    ys = rs * np.sin(theta)

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines",
        line=dict(color=MASAI_RED, width=3),
        fill="toself", fillcolor="rgba(237,28,36,0.10)",
        showlegend=False,
        hovertemplate=f"l1_ratio={rho}<br>β₁=%{{x:.2f}}, β₂=%{{y:.2f}}<extra></extra>",
    ), row=1, col=col)

    fig.add_hline(y=0, line_color="#ccc", line_width=1, row=1, col=col)
    fig.add_vline(x=0, line_color="#ccc", line_width=1, row=1, col=col)
    fig.update_xaxes(range=[-1.3, 1.3], scaleanchor=f"y{col}", scaleratio=1, row=1, col=col)
    fig.update_yaxes(range=[-1.3, 1.3], row=1, col=col)

fig.update_layout(
    title="The ElasticNet constraint region morphs from diamond → rounded → circle",
    paper_bgcolor="white", plot_bgcolor="white",
    width=1050, height=420,
)
fig.show()


In [ ]:
box("output", "Reading the morph",
    "At <b>l1_ratio = 1.0</b> (left), corners stick out sharply on the axes — incoming loss contours touch corners → exact zeros (Lasso behaviour).<br><br>"
    "At <b>l1_ratio = 0.5</b> (middle), the corners are rounded off but still pointier than the edges — there's <i>some</i> bias toward sparsity, but coefficients near zero may not zero out cleanly. This is ElasticNet's compromise: a little less sparsity, a lot more stability when features are correlated.<br><br>"
    "At <b>l1_ratio = 0.0</b> (right), the shape is a perfect circle — no corners, no sparsity. Pure Ridge behaviour.")


In [ ]:
# Three ElasticNet variants
for l1r in [0.1, 0.5, 0.9]:
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("en", ElasticNet(alpha=0.5, l1_ratio=l1r, max_iter=10_000))])
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps["en"].coef_
    n_nonzero = sum(coefs != 0)
    test_r2 = pipe.score(X_test, y_test)
    print(f"l1_ratio={l1r}: non-zero = {n_nonzero}/10, test R² = {test_r2:.3f}")


l1_ratio=0.1: non-zero = 10/10, test R² = 0.456
l1_ratio=0.5: non-zero = 10/10, test R² = 0.462
l1_ratio=0.9: non-zero = 10/10, test R² = 0.463


In [ ]:
box("industry", "When to reach for ElasticNet",
    "Use ElasticNet over Lasso when (a) you expect correlated features, or (b) Lasso's selected features are unstable across reruns. "
    "Common pattern: tune (alpha, l1_ratio) jointly with GridSearchCV.")


## Part 2F — Alpha tuning via cross-validation

The synthesis moment: regularisation needs CV to pick alpha. sklearn provides convenience classes that internally do exactly the alpha-sweep + CV loop we've been doing manually.


In [ ]:
box("warning", "Picking alpha by hand (or by training R²) is a Kaggle-losing mistake",
    "Training R² <b>monotonically decreases</b> as alpha grows — less constraint = better training fit. Picking alpha to maximise training R² always gives α = 0, which is plain OLS, which is what we're trying to escape.<br><br>"
    "The right metric is <b>CV R²</b> (or CV MSE). Sweep alpha across a log scale, compute CV R² at each, pick the alpha where CV R² peaks. <b>That's the bias-variance sweet spot for this dataset.</b>")


### CV R² vs alpha — the curve that picks alpha for you

Before reaching for `RidgeCV` and `LassoCV` (the convenience classes), let's see what they're really doing under the hood: at each alpha, run K-fold CV, average the scores, plot the curve. The peak of the curve is the best alpha.


In [ ]:
# Manual alpha sweep with CV for Ridge AND Lasso
alphas_sweep_ridge = np.logspace(-2, 4, 30)
alphas_sweep_lasso = np.logspace(-3, 1, 30)
cv_obj = KFold(n_splits=5, shuffle=True, random_state=RNG)

ridge_cv_mean, ridge_cv_std = [], []
for a in alphas_sweep_ridge:
    pipe = Pipeline([("scaler", StandardScaler()), ("m", Ridge(alpha=a))])
    s = cross_val_score(pipe, X_train, y_train, cv=cv_obj, scoring="r2")
    ridge_cv_mean.append(s.mean()); ridge_cv_std.append(s.std())
ridge_cv_mean = np.array(ridge_cv_mean); ridge_cv_std = np.array(ridge_cv_std)

lasso_cv_mean, lasso_cv_std = [], []
for a in alphas_sweep_lasso:
    pipe = Pipeline([("scaler", StandardScaler()), ("m", Lasso(alpha=a, max_iter=10_000))])
    s = cross_val_score(pipe, X_train, y_train, cv=cv_obj, scoring="r2")
    lasso_cv_mean.append(s.mean()); lasso_cv_std.append(s.std())
lasso_cv_mean = np.array(lasso_cv_mean); lasso_cv_std = np.array(lasso_cv_std)

best_ridge_alpha_manual = alphas_sweep_ridge[int(np.argmax(ridge_cv_mean))]
best_lasso_alpha_manual = alphas_sweep_lasso[int(np.argmax(lasso_cv_mean))]
print(f"Best Ridge alpha (manual sweep): {best_ridge_alpha_manual:.4f}, peak CV R² = {ridge_cv_mean.max():.3f}")
print(f"Best Lasso alpha (manual sweep): {best_lasso_alpha_manual:.4f}, peak CV R² = {lasso_cv_mean.max():.3f}")


Best Ridge alpha (manual sweep): 0.7279, peak CV R² = 0.481
Best Lasso alpha (manual sweep): 0.5736, peak CV R² = 0.481


In [ ]:
# Plot CV R² (mean + std band) vs alpha for both models
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=(
                        f"Ridge — best alpha = {best_ridge_alpha_manual:.3f}",
                        f"Lasso — best alpha = {best_lasso_alpha_manual:.3f}",
                    ),
                    horizontal_spacing=0.06)

# Ridge panel
fig.add_trace(go.Scatter(x=alphas_sweep_ridge, y=ridge_cv_mean + ridge_cv_std,
                         mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=alphas_sweep_ridge, y=ridge_cv_mean - ridge_cv_std,
                         mode="lines", line=dict(width=0),
                         fill="tonexty", fillcolor="rgba(21,101,192,0.18)",
                         showlegend=False, hoverinfo="skip"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=alphas_sweep_ridge, y=ridge_cv_mean,
                         mode="lines+markers", name="Ridge CV R²",
                         line=dict(color=BLUE, width=3),
                         hovertemplate="alpha=%{x:.3f}<br>CV R²=%{y:.3f}<extra></extra>"),
              row=1, col=1)
fig.add_vline(x=best_ridge_alpha_manual, line_dash="dash", line_color=MASAI_RED, row=1, col=1)

# Lasso panel
fig.add_trace(go.Scatter(x=alphas_sweep_lasso, y=lasso_cv_mean + lasso_cv_std,
                         mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"),
              row=1, col=2)
fig.add_trace(go.Scatter(x=alphas_sweep_lasso, y=lasso_cv_mean - lasso_cv_std,
                         mode="lines", line=dict(width=0),
                         fill="tonexty", fillcolor="rgba(237,28,36,0.18)",
                         showlegend=False, hoverinfo="skip"),
              row=1, col=2)
fig.add_trace(go.Scatter(x=alphas_sweep_lasso, y=lasso_cv_mean,
                         mode="lines+markers", name="Lasso CV R²",
                         line=dict(color=MASAI_RED, width=3),
                         hovertemplate="alpha=%{x:.3f}<br>CV R²=%{y:.3f}<extra></extra>"),
              row=1, col=2)
fig.add_vline(x=best_lasso_alpha_manual, line_dash="dash", line_color=MASAI_RED, row=1, col=2)

fig.update_xaxes(type="log", title_text="alpha (log)", row=1, col=1)
fig.update_xaxes(type="log", title_text="alpha (log)", row=1, col=2)
fig.update_yaxes(title_text="CV R² (mean ± std)", row=1, col=1)

fig.update_layout(
    title="CV R² vs alpha — the peak picks alpha; the shaded band is the CV uncertainty",
    paper_bgcolor="white", plot_bgcolor="white",
    width=1100, height=480,
)
fig.show()


In [ ]:
box("output", "Reading the CV-vs-alpha curve",
    f"Both curves rise from a flat-bias regime on the left (too little regularisation; effectively OLS) to a peak, then fall as alpha grows too large (too much regularisation; everything shrinks toward zero).<br><br>"
    f"<b>Ridge peak:</b> alpha ≈ {best_ridge_alpha_manual:.3f}, CV R² ≈ {ridge_cv_mean.max():.3f}.<br>"
    f"<b>Lasso peak:</b> alpha ≈ {best_lasso_alpha_manual:.3f}, CV R² ≈ {lasso_cv_mean.max():.3f}.<br><br>"
    "The shaded band is the CV std across the 5 folds. Where the band is wide, the CV estimate is noisy — be wary of trusting a peak inside the noise. Where it's tight, the peak is reliable.<br><br>"
    "`RidgeCV` and `LassoCV` below do exactly this sweep, just faster (they exploit problem-specific algebra to avoid retraining from scratch at each alpha).")


In [ ]:
# RidgeCV - efficient, built-in CV
ridge_cv_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge_cv", RidgeCV(alphas=np.logspace(-3, 4, 100), cv=5)),
])
ridge_cv_pipe.fit(X_train, y_train)
best_ridge_alpha = ridge_cv_pipe.named_steps["ridge_cv"].alpha_
print(f"Best Ridge alpha: {best_ridge_alpha:.4f}")
print(f"Test R²: {ridge_cv_pipe.score(X_test, y_test):.3f}")


Best Ridge alpha: 39.4421
Test R²: 0.460


In [ ]:
# LassoCV - efficient, built-in CV
lasso_cv_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso_cv", LassoCV(alphas=np.logspace(-3, 1, 100), cv=5, max_iter=10_000, random_state=RNG)),
])
lasso_cv_pipe.fit(X_train, y_train)
best_lasso_alpha = lasso_cv_pipe.named_steps["lasso_cv"].alpha_
lasso_cv_coefs = lasso_cv_pipe.named_steps["lasso_cv"].coef_
n_nonzero_cv = sum(lasso_cv_coefs != 0)
print(f"Best Lasso alpha: {best_lasso_alpha:.4f}")
print(f"Non-zero features at best alpha: {n_nonzero_cv}/10")
print(f"Test R²: {lasso_cv_pipe.score(X_test, y_test):.3f}")


Best Lasso alpha: 1.5557
Non-zero features at best alpha: 7/10
Test R²: 0.471


In [ ]:
# GridSearchCV for ElasticNet's 2D grid
en_pipe = Pipeline([("scaler", StandardScaler()),
                    ("en", ElasticNet(max_iter=10_000, random_state=RNG))])
param_grid = {
    "en__alpha":    np.logspace(-3, 1, 20),
    "en__l1_ratio": np.linspace(0.1, 1.0, 10),
}
gs = GridSearchCV(en_pipe, param_grid,
                  cv=KFold(n_splits=5, shuffle=True, random_state=RNG),
                  scoring="r2", n_jobs=-1)
gs.fit(X_train, y_train)
print(f"Best ElasticNet params: {gs.best_params_}")
print(f"Best CV R²: {gs.best_score_:.3f}")
print(f"Test R²: {gs.score(X_test, y_test):.3f}")


Best ElasticNet params: {'en__alpha': np.float64(0.5455594781168515), 'en__l1_ratio': np.float64(1.0)}
Best CV R²: 0.481
Test R²: 0.461


### Visualise the GridSearchCV result as a 2D heatmap


In [ ]:
# Extract CV scores from GridSearchCV results
results = pd.DataFrame(gs.cv_results_)
score_matrix = results.pivot_table(
    values="mean_test_score",
    index="param_en__l1_ratio",
    columns="param_en__alpha",
)
fig = go.Figure(data=go.Heatmap(
    z=score_matrix.values,
    x=[f"{a:.3f}" for a in score_matrix.columns],
    y=[f"{l:.2f}" for l in score_matrix.index],
    colorscale="Viridis",
    hovertemplate="alpha = %{x}<br>l1_ratio = %{y}<br>CV R² = %{z:.3f}<extra></extra>",
))
fig.update_layout(
    title="ElasticNet GridSearchCV — CV R² across (alpha, l1_ratio)",
    xaxis_title="alpha",
    yaxis_title="l1_ratio",
    paper_bgcolor="white", plot_bgcolor="white",
    width=800, height=500,
)
fig.show()


## Part 2G — Final comparison: all four models on the test set

**Pedagogical pattern: multiple wrong approaches culminating in the best.**


In [ ]:
# Test set evaluation across all four
final_results = []

ols_pipe.fit(X_train, y_train)
final_results.append(("OLS", ols_pipe.score(X_test, y_test),
                      mean_squared_error(y_test, ols_pipe.predict(X_test)),
                      sum(ols_pipe.named_steps["model"].coef_ != 0)))

final_results.append(("Ridge (CV-tuned)", ridge_cv_pipe.score(X_test, y_test),
                      mean_squared_error(y_test, ridge_cv_pipe.predict(X_test)),
                      sum(ridge_cv_pipe.named_steps["ridge_cv"].coef_ != 0)))

final_results.append(("Lasso (CV-tuned)", lasso_cv_pipe.score(X_test, y_test),
                      mean_squared_error(y_test, lasso_cv_pipe.predict(X_test)),
                      sum(lasso_cv_pipe.named_steps["lasso_cv"].coef_ != 0)))

best_en_pipe = gs.best_estimator_
final_results.append(("ElasticNet (grid-tuned)", best_en_pipe.score(X_test, y_test),
                      mean_squared_error(y_test, best_en_pipe.predict(X_test)),
                      sum(best_en_pipe.named_steps["en"].coef_ != 0)))

results_df = pd.DataFrame(final_results, columns=["Model", "Test R²", "Test MSE", "Non-zero coefs"])
results_df


,Model,Test R²,Test MSE,Non-zero coefs
0,OLS,0.452603,2900.193628,10
1,Ridge (CV-tuned),0.460477,2858.476319,10
2,Lasso (CV-tuned),0.471280,2801.239596,7
3,ElasticNet (grid-tuned),0.461305,2854.088842,9


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=results_df["Model"], y=results_df["Test R²"], name="Test R²",
                     marker_color=BLUE,
                     text=[f"{v:.3f}" for v in results_df["Test R²"]],
                     textposition="outside",
                     hovertemplate="%{x}<br>Test R² = %{y:.3f}<extra></extra>"))
fig.update_layout(
    title="Final test-set R² across all four models",
    yaxis=dict(range=[0, 0.6], gridcolor="#eee"),
    paper_bgcolor="white", plot_bgcolor="white",
    width=750, height=450,
)
fig.show()


In [ ]:
best_row = results_df.iloc[results_df["Test R²"].idxmax()]
spread = results_df["Test R²"].max() - results_df["Test R²"].min()
box("output", "The numerical winner is...",
    f"<b>{best_row['Model']}</b> with test R² = {best_row['Test R²']:.3f}.<br><br>"
    f"BUT — spread across all 4 models is only {spread:.3f}. "
    "On a 442-row dataset with CV stdev ~0.05, differences this small are within noise. "
    "Picking based on test R² alone is overfitting to the specific test split. "
    "Consider interpretability (Lasso/ElasticNet drop features) and stability (Ridge/ElasticNet are stable on correlated features) when choosing.")


## Part 2H — Lasso for feature selection

Lasso's sparsity is automatic feature selection. `SelectFromModel` formalises this.


In [ ]:
# Fit LassoCV on scaled training data
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_cv = LassoCV(alphas=np.logspace(-3, 1, 100), cv=5, max_iter=10_000, random_state=RNG).fit(X_train_scaled, y_train)
selector = SelectFromModel(lasso_cv, prefit=True)
X_train_sel = selector.transform(X_train_scaled)

selected = [name for name, sel in zip(feature_names, selector.get_support()) if sel]
dropped = [name for name, sel in zip(feature_names, selector.get_support()) if not sel]
print(f"Lasso best alpha:  {lasso_cv.alpha_:.4f}")
print(f"Selected features ({len(selected)}/10): {selected}")
print(f"Dropped features ({len(dropped)}/10):  {dropped}")


Lasso best alpha:  1.5557
Selected features (7/10): ['sex', 'bmi', 'bp', 's1', 's3', 's5', 's6']
Dropped features (3/10):  ['age', 's2', 's4']


In [ ]:
box("warning", "Lasso's feature selection is unstable on correlated features",
    "With the s1-s6 correlations on Diabetes, the SPECIFIC subset Lasso picks can shift between runs. "
    "For research papers and production: report the surviving set with a stability measure (run with 100 different seeds; report features that survive in >80% of runs).")


---

## Ten takeaways

1. Every model's expected error is bias² + variance + irreducible noise. You control two of three.
2. Training error monotonically decreases with complexity; test error has a U-shape. The minimum is the sweet spot.
3. A single train/test split's R² is one sample from a noisy distribution. Use cross-validation.
4. `KFold(shuffle=True, random_state=42)` is the canonical default for regression. `StratifiedKFold` is for classification.
5. Always wrap preprocessing in a `Pipeline` and pass the Pipeline to `cross_val_score`. Otherwise: leakage.
6. Ridge (L2) shrinks all coefficients smoothly toward zero. No exact zeros. Geometry: circle constraint.
7. Lasso (L1) drives some coefficients to exactly zero. Sparsity is automatic feature selection. Geometry: diamond constraint.
8. ElasticNet (L1 + L2) compromises — useful when features are correlated and Lasso is unstable.
9. Pick alpha by CV, not by training R². `RidgeCV`, `LassoCV`, and `GridSearchCV` are the tools.
10. Small test-R² differences (< CV stdev) are noise. Choose by interpretability and stability when generalisation is similar.

---

Vishlesan i-Hub IIT Patna × Masai School
